# Version 3
New features:
- Learning rate schedulers
- Training and validation metrics

## Imports

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import matplotlib.mticker as mticker


In [ ]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Helper functions

In [ ]:
def plot_training_metrics(metrics):
    """
    Plots the training and validation metrics from a model training process.

    This function generates two side-by-side plots:
    1. Training Loss vs. Validation Loss.
    2. Validation Accuracy.

    Args:
        metrics (list): A list or tuple containing three lists:
                        [train_losses, val_losses, val_accuracies].
    """
    # Unpack the metrics into their respective lists
    train_losses, val_losses, val_accuracies = metrics
    
    # Determine the number of epochs from the length of the training losses list
    num_epochs = len(train_losses)
    # Create a 1-indexed range of epoch numbers for the x-axis
    epochs = range(1, num_epochs + 1)

    # Create a figure and a set of subplots with 1 row and 2 columns
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # --- Configure the first subplot for training and validation loss ---
    # Select the first subplot
    ax1 = axes[0]
    # Plot training loss data
    ax1.plot(epochs, train_losses, color='#085c75', linewidth=2.5, marker='o', markersize=5, label='Training Loss')
    # Plot validation loss data
    ax1.plot(epochs, val_losses, color='#fa5f64', linewidth=2.5, marker='o', markersize=5, label='Validation Loss')
    # Set the title and axis labels for the loss plot
    ax1.set_title('Training & Validation Loss', fontsize=14)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    # Display the legend
    ax1.legend()
    # Add a grid for better readability
    ax1.grid(True, linestyle='--', alpha=0.6)

    # --- Configure the second subplot for validation accuracy ---
    # Select the second subplot
    ax2 = axes[1]
    # Plot validation accuracy data
    ax2.plot(epochs, val_accuracies, color='#fa5f64', linewidth=2.5, marker='o', markersize=5, label='Validation Accuracy')
    # Set the title and axis labels for the accuracy plot
    ax2.set_title('Validation Accuracy', fontsize=14)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    # Display the legend
    ax2.legend()
    # Add a grid for better readability
    ax2.grid(True, linestyle='--', alpha=0.6)
    
    # --- Apply dynamic and consistent styling to both subplots ---
    # Calculate a suitable interval for the x-axis ticks to avoid clutter
    x_interval = (num_epochs - 1) // 10 + 1

    # Loop through each subplot to apply common axis settings
    for ax in axes:
        # Set the y-axis to start at 0 and the x-axis to span the epochs
        ax.set_ylim(bottom=0)
        ax.set_xlim(left=1, right=num_epochs)
        
        # Set the major tick locator for the x-axis using the dynamic interval
        ax.xaxis.set_major_locator(mticker.MultipleLocator(x_interval))
        # Set the font size for the tick labels on both axes
        ax.tick_params(axis='both', which='major', labelsize=10)

    # Adjust subplot parameters for a tight layout
    plt.tight_layout()
    # Display the plots
    plt.show()

## Data prepping

In [ ]:
# Define data paths
data_paths = ['./data']  # TODO: Update with actual data paths

In [ ]:
# Create datasets (if applicable)
## Inspect data
# Get sample and label from 0th index
# Print shapes and types
# Print labels and PV locations

In [ ]:
# Visualize sample and corresponding label (create in utils)

In [ ]:
# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

## Create the model
This will be the simplest form of the model possible, no bells and whistles, 
completely hardcoded.

In [ ]:
class CNNBlock(nn.Module):
    """
    Defines a single convolutional block for a CNN.

    This block consists of a convolutional layer, batch normalization,
    a ReLU activation, and a max-pooling layer, bundled as a sequential module.
    """
    def __init__(self, in_channels, out_channels, kernel_size=3):
        """
        Initializes the layers of the CNNBlock.

        Args:
            in_channels (int): Number of channels in the input image.
            out_channels (int): Number of channels produced by the convolution.
            kernel_size (int, optional): Size of the convolving kernel. Defaults to 3.
            padding (int, optional): Zero-padding added to both sides of the input. Defaults to 1.
        """
        super(CNNBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=kernel_size, 
                      padding=(kernel_size-1)//2),  # same padding,
            nn.BatchNorm1d(out_channels),
            nn.LeakyReLU()  # Could start with ReLU instead and switch to LeakyReLU later
            # Can't do MaxPool1d because we need equal sized input and output
            # nn.MaxPool1d(kernel_size=2, stride=2)
        )
    
    def forward(self, x):
        """
        Defines the forward pass of the CNNBlock.

        Args:
            x (torch.Tensor): Input tensor of shape (batch_size, in_channels, length).

        Returns:
            torch.Tensor: Output tensor after applying the convolutional block.
        """
        return self.block(x)

In [ ]:
# Verify the CNNBlock
print("--- Verifying CNNBlock ---\n")

# Instantiate the block with 1 input channel and 1 output channel
verify_cnn_block = CNNBlock(in_channels=1, out_channels=1)
print("Block Structure:\n")
print(verify_cnn_block)

# Verify the output shape after a forward pass
# Create a dummy input tensor (batch_size=1, channels=1, length=4000), i.e.,
# a single 4000 bin histogram
dummy_input = torch.randn(1, 1, 4000)
print(f"\nInput tensor shape:  {dummy_input.shape}")

# Pass the dummy tensor through the block
output = verify_cnn_block(dummy_input)
print(f"Output tensor shape: {output.shape}")

In [ ]:
# Assemble the full model
class SimpleCNN(nn.Module):
    def __init__(self):
        # Initialize parent nn.Module class
        super(SimpleCNN, self).__init__()

        self.conv1_block = CNNBlock(1, 16)
        self.conv2_block = CNNBlock(16, 32)
        self.conv3_block = CNNBlock(32, 64)
        self.conv4_block = CNNBlock(64, 64)
        self.conv5_block = CNNBlock(64, 64)

        self.head = nn.Conv1d(64, 1, kernel_size=1)

    def forward(self, x):
        x = self.conv1_block(x)
        x = self.conv2_block(x)
        x = self.conv3_block(x)
        x = self.conv4_block(x)
        x = self.conv5_block(x)

        # Pass through the locator head
        x = self.head(x)  # (batch_size, 1, 4000)
        return x.squeeze(1)  # (batch_size, 4000)


In [ ]:
# Verify the SimpleCNN
print("--- Verifying SimpleCNN ---\n")

# Verify the structure of the model
verify_simple_cnn = SimpleCNN()
print("Model Structure:\n")
print(verify_simple_cnn)

# Verify the output shape after a forward pass
# Create a dummy input tensor (batch_size=64, channels=1, length=4000)
dummy_input = torch.randn(64, 1, 4000)
print(f"\nInput tensor shape:  {dummy_input.shape}")

# Pass the dummy tensor through the model
output = verify_simple_cnn(dummy_input)
print(f"Output tensor shape: {output.shape}")

In [ ]:
# Create model
model = SimpleCNN()
n_epochs = 100  # TODO: placeholder value

# Loss function
# TODO: understand why we use a custom cost function instead of a standard one
loss_function = None

# Optimizer
# TODO: weight_decay=5e-4
# TODO: learning rate scheduler (e.g., StepLR, ReduceLROnPlateau, 
# CosineAnnealingLR)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Learning rate scheduler
# Cosine annealing scheduler, which reduces learning rate smoothly following a 
# cosine curve from initial learning rate value to minimum over specified epochs
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs,
                                                 eta_min=1e-6)

In [ ]:
# TODO: untangle mess model loops scattered all throughout pv-finder, in loss.py
# / alt_loss_A.py, training.py, utilities.py (progress bar, etc.), device
# allocation, epoch loop, train loop, validation loop
#
# I will sketch out the training loop below, but it will need to be refactored
# for pv-finder's specific needs and to ensure clean separation of concerns.

In [ ]:
# Define training loop for one epoch (one pass through the training data)
def train_epoch(model, loss_function, optimizer, train_loader, device):
    model.to(device)  # Move model to the appropriate device
    model.train()  # Set model to training mode
    total_loss = 0
    n_correct_predictions, n_total_predictions = 0, 0

    for inputs, targets in train_loader:
        # Move data to the appropriate device
        inputs, targets = inputs.to(device), targets.to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)

        # Compute loss
        loss = loss_function(outputs, targets)

        # Backpropagation to compute gradients
        loss.backward()

        # Update parameters
        optimizer.step()

        # Accumulate total loss
        total_loss += loss.item()

        # Calculate efficiency metrics
        # TODO: Understand Simon's implementation

        # Count correct predictions
        # TODO: This is a placeholder for calculating correct predictions
        predicted_indices = outputs.argmax(dim=1)
        correct_predictions = predicted_indices.eq(targets)
        n_correct_in_batch = correct_predictions.sum().item()
        # Accumulate total correct predictions
        n_correct_predictions += n_correct_in_batch

        # Accumulate total predictions
        batch_size = len(targets)
        n_total_predictions += batch_size

    accuracy = n_correct_predictions / n_total_predictions
    avg_loss = total_loss / len(train_loader)
    print(f"Epoch loss (avg.): {avg_loss:.3f} | Accuracy: {accuracy*100:.2f}%")

    return model, total_loss

In [ ]:
# Define one validation loop
def evaluate(model, val_loader, device):
    model.eval()  # Set model to evaluation mode
    n_correct, n_total = 0, 0
    total_val_loss = 0

    with torch.no_grad():  # Disable gradient calculation for validation
        for inputs, targets in val_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)

            # Calculate validation loss for batch
            val_loss = loss_function(outputs, targets)
            # Accumulate validation loss
            total_val_loss += val_loss.item()

            # Count correct predictions
            predicted_indices = outputs.argmax(dim=1)
            correct_predictions = predicted_indices.eq(targets)
            n_correct_in_batch = correct_predictions.sum().item()
            n_correct += n_correct_in_batch

            # Count total predictions
            batch_size = len(targets)
            n_total += batch_size
        
        # Calculate accuracy
        accuracy = n_correct / n_total
        avg_val_loss = total_val_loss / len(val_loader)
        print(f"Validation loss (avg.): {avg_val_loss:.3f} | Validation accuracy: {accuracy*100:.2f}%")
    
    return accuracy, total_val_loss

In [ ]:
# Train and evaluate the model for a number of epochs
def train_and_evaluate(model, loss_function, optimizer, train_loader, 
                       val_loader, device, n_epochs, scheduler=None):
    train_losses, val_losses, val_accuracies = [], [], []
    for epoch in range(n_epochs):
        print(f"Epoch {epoch + 1}/{n_epochs}")
        trained_model, train_loss = train_epoch(model, loss_function, optimizer,
                                                train_loader, device)
        val_accuracy, val_loss = evaluate(model, val_loader, device)

        # Call step scheduler if there is one present
        if scheduler is not None: scheduler.step()

        # Store metrics
        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_accuracies.append(val_accuracy)
    
    metrics = [train_losses, val_losses, val_accuracies]
    # Return trained model and metrics for plotting
    return trained_model, metrics


In [ ]:
trained_model, metrics = train_and_evaluate(model=model, 
                                   loss_function=loss_function, 
                                   optimizer=optimizer, 
                                   train_loader=train_loader, 
                                   val_loader=val_loader, 
                                   device=device, 
                                   n_epochs=n_epochs,
                                   scheduler=scheduler)

In [ ]:
# Visualize the training metrics for the full model
print("\n--- Training Plots ---\n")
plot_training_metrics(metrics)